In [ ]:
# custom VQGAN architecture, inspired to https://github.com/compvis/taming-transformers
# and tailored for the texture synthesis task on the DTD dataset https://www.robots.ox.ac.uk/~vgg/data/dtd/$0

In [ ]:
import os
from PIL import Image
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
import torch
import torch.optim as optim
from torchvision.models import vgg16
import matplotlib.pyplot as plt
import time

VQGAN:
- Generator: Encoder-VQ-Decoder. It takes a real texture image and tries to generate a flawless, high-resolution copy of it
- Discriminator: it looks at the original image and the generated copy, trying to figure out which one is the fake one

Perceptual Loss (Reconstruction Loss): LPIPS (Learned Perceptual Image Patch Similarity). It does not look at the raw pixel coords, but the real texture and the generated texture through a pre-trained net that knows how to recognize shapes, edges and textures. It compares the hidden feature layers (it looks at the same freq of lines, same style of roughness, etc.)

VGG16: we freeze weights because when Perceptual Loss is high and the gradient flows backwards, only the Generator is updates. It's a pretrained net, we are only using its specialized vision

$Loss_{VQGAN} = L_{recon} + \lambda \cdot L_{GAN} + L_{percep} + L_{codebook}$

where

- $L_{GAN}$ uses the Hinge Loss for both generator and discriminator, with adaptive weighting: $\lambda = \frac{\nabla _{G_{L}} [L_{recon} + L_{percep}]}{\nabla _{G_{L}} [L_{GAN} + ϵ]}$


In [ ]:
class Encoder(nn.Module):
    def __init__(self, hidden_dim=128, embedding_dim=256):
        super().__init__()
        self.conv1 = nn.Conv2d(3, hidden_dim // 2, kernel_size=4, stride=2, padding=1)              # 512x512 -> 256x256
        self.conv2 = nn.Conv2d(hidden_dim // 2, hidden_dim, kernel_size=4, stride=2, padding=1)     # 256x256 -> 128x128
        self.conv3 = nn.Conv2d(hidden_dim, hidden_dim * 2, kernel_size=4, stride=2, padding=1)      # 128x128 -> 64x64
        self.downsample = nn.Conv2d(hidden_dim * 2, embedding_dim, kernel_size=4, stride=4, padding=0)  # 64x64 -> 16x16

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = self.downsample(x)
        return x

class Decoder(nn.Module):
    def __init__(self, embedding_dim=256, hidden_dim=128):
        super().__init__()
        self.upsample = nn.ConvTranspose2d(embedding_dim, hidden_dim * 2, kernel_size=4, stride=4, padding=0) #16x16 -> 64x64
        self.conv1 = nn.ConvTranspose2d(hidden_dim * 2, hidden_dim, kernel_size=4, stride=2, padding=1)       # 64x64 -> 128x128
        self.conv2 = nn.ConvTranspose2d(hidden_dim, hidden_dim // 2, kernel_size=4, stride=2, padding=1)      # 128x128 -> 256x256
        self.conv3 = nn.ConvTranspose2d(hidden_dim // 2, 3, kernel_size=4, stride=2, padding=1)               # 256x256 -> 512x512

    def forward(self, x):
        x = self.upsample(x)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.conv3(x)
        x = torch.tanh(x)
        return x

class VectorQuantizer(nn.Module):
    def __init__(self, num_embeddings=1024, embedding_dim=256, commitment_cost=0.25):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.num_embeddings = num_embeddings
        self.commitment_cost = commitment_cost

        self.embeddings = nn.Embedding(num_embeddings, embedding_dim)
        self.embeddings.weight.data.uniform_(-1/self.num_embeddings, 1/self.num_embeddings)

    def forward(self, z):
        z_flattened = z.permute(0, 2, 3, 1).contiguous()
        z_flattened = z_flattened.view(-1, self.embedding_dim)

        distances = (torch.sum(z_flattened**2, dim=1, keepdim=True)
                     + torch.sum(self.embeddings.weight**2, dim=1)
                     - 2 * torch.matmul(z_flattened, self.embeddings.weight.t()))

        encoding_indices = torch.argmin(distances, dim=1)
        encodings = F.one_hot(encoding_indices, self.num_embeddings).float()

        quantized = torch.matmul(encodings, self.embeddings.weight)
        quantized = quantized.view(z.shape[0], z.shape[2], z.shape[3], self.embedding_dim)
        quantized = quantized.permute(0, 3, 1, 2).contiguous()

        e_latent_loss = F.mse_loss(quantized.detach(), z)
        q_latent_loss = F.mse_loss(quantized, z.detach())
        loss = q_latent_loss + self.commitment_cost * e_latent_loss

        quantized = z + (quantized - z).detach()

        avg_probs = torch.mean(encodings, dim=0)
        perplexity = torch.exp(-torch.sum(avg_probs * torch.log(avg_probs + 1e-10)))

        active_codes = torch.unique(encoding_indices).numel()

        return quantized, loss, perplexity, active_codes

In [ ]:
class VQGAN(nn.Module):
    def __init__(self, hidden_dim=128, embedding_dim=256, num_embeddings=1024, commitment_cost=0.25):
        super().__init__()
        self.encoder = Encoder(hidden_dim, embedding_dim)
        self.vq = VectorQuantizer(num_embeddings, embedding_dim, commitment_cost)
        self.decoder = Decoder(embedding_dim, hidden_dim)

    def forward(self, x):
        z = self.encoder(x)
        quantized, vq_loss, perplexity, active_codes = self.vq(z)
        x_recon = self.decoder(quantized)
        return x_recon, vq_loss, perplexity, active_codes

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, hidden_dim=64):
        super().__init__()
        self.conv1 = nn.Conv2d(3, hidden_dim, kernel_size=4, stride=2, padding=1)                   # (3, 256, 256) -> (64, 128, 128)
        self.relu = nn.LeakyReLU(negative_slope=0.2, inplace=True)

        self.conv2 = nn.Conv2d(hidden_dim, hidden_dim * 2, kernel_size=4, stride=2, padding=1)      # (64, 128, 128) -> (128, 64, 64)
        self.norm1 = nn.BatchNorm2d(hidden_dim * 2)

        self.conv3 = nn.Conv2d(hidden_dim * 2, hidden_dim * 4, kernel_size=4, stride=2, padding=1)  # (128, 64, 64) -> (256, 32, 32)
        self.norm2 = nn.BatchNorm2d(hidden_dim * 4)

        self.conv4 = nn.Conv2d(hidden_dim * 4, 1, kernel_size=4, stride=1, padding=1)               # (256, 32, 32) -> (1, 31, 31)

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.relu(self.norm1(self.conv2(x)))
        x = self.relu(self.norm2(self.conv3(x)))
        x = self.conv4(x)
        return x

In [ ]:
class PerceptualLoss(nn.Module):
    def __init__(self):
        super().__init__()
        VGG16 = vgg16(weights='DEFAULT').features
        self.slice = nn.Sequential(*list(VGG16.children())[:16]).eval()
        for param in self.slice.parameters():
            param.requires_grad = False

    def forward(self, real, fake):
        return F.mse_loss(self.slice(fake), self.slice(real))

====> Epoch 0 Finished | Avg G-Loss: 1.7020 | Avg D-Loss: 0.0000
      Generator ========> Average Loss:1.7020 | Perplexity: 2.42 | Active codes: 4.61
      Discriminator ====> Average Loss:0.0000

====> Epoch 1 Finished | Avg G-Loss: 1.0992 | Avg D-Loss: 0.0000
      Generator ========> Average Loss:1.0992 | Perplexity: 3.68 | Active codes: 7.03
      Discriminator ====> Average Loss:0.0000

====> Epoch 2 Finished | Avg G-Loss: 0.8506 | Avg D-Loss: 0.8715
      Generator ========> Average Loss:0.8506 | Perplexity: 5.27 | Active codes: 8.91
      Discriminator ====> Average Loss:0.8715

====> Epoch 3 Finished | Avg G-Loss: 0.7840 | Avg D-Loss: 0.9989
      Generator ========> Average Loss:0.7840 | Perplexity: 6.17 | Active codes: 11.59
      Discriminator ====> Average Loss:0.9989

====> Epoch 4 Finished | Avg G-Loss: 0.7576 | Avg D-Loss: 1.0018
      Generator ========> Average Loss:0.7576 | Perplexity: 6.67 | Active codes: 12.73
      Discriminator ====> Average Loss:1.0018

====> Ep